# Tugas — Python Informed Search

**Nama : Julianda Caesar Prakoso**

**NRP : 5025251028**

**Kelas : KKA C**

---
## Kasus Penyelesaian 8-Puzzle
Penylesaian **8-Puzzle** menggunakan dua algoritma:
1. **Greedy Best First Search (GBFS)** menggunakan nilai heuristik `h(n)`.
2. **A\*** menggunakan `f(n) = g(n) + h(n)`.

Konfigurasi **initial state** dan **goal state** dapat diinputkan oleh user secara bebas. Setiap perpindahan kotak dihitung sebagai **actual cost = 1**, sedangkan fungsi heuristik yang digunakan adalah **Manhattan Distance**, yaitu jumlah jarak setiap angka dari posisi saat ini menuju posisi tujuan.

Kotak kosong direpresentasikan dengan angka 0. Program menyediakan antarmuka untuk memilih algoritma, menjalankan pencarian, serta menampilkan **rute solusi** dan **total cost** hasil pencarian.

### Fungsi Heuristik: Manhattan Distance
Rumus:

`h(n) = Σ |x_current - x_goal| + |y_current - y_goal|`

Kotak kosong (0) tidak dihitung.

In [19]:
import heapq

def manhattan_distance(state, goal):
    distance = 0

    for i in range(9):
        tile = state[i]
        if tile == 0:
            continue

        current_row = i // 3
        current_col = i % 3

        goal_index = goal.index(tile)
        goal_row = goal_index // 3
        goal_col = goal_index % 3

        distance += abs(current_row - goal_row)
        distance += abs(current_col - goal_col)

    return distance

### Generate Neighbor
Kotak kosong (0) dapat berpindah ke posisi yang bersebelahan secara
horizontal atau vertikal.

Kemungkinan gerakan:
- Atas
- Bawah
- Kiri
- Kanan

Setiap gerakan memiliki **cost = 1**.


In [20]:
def get_neighbors(state):
    neighbors = []

    zero_index = state.index(0)

    row = zero_index // 3
    col = zero_index % 3

    # Atas
    if row > 0:
        new_state = list(state)
        swap_index = zero_index - 3
        new_state[zero_index], new_state[swap_index] = new_state[swap_index], new_state[zero_index]
        neighbors.append(tuple(new_state))

    # Bawah
    if row < 2:
        new_state = list(state)
        swap_index = zero_index + 3
        new_state[zero_index], new_state[swap_index] = new_state[swap_index], new_state[zero_index]
        neighbors.append(tuple(new_state))

    # Kiri
    if col > 0:
        new_state = list(state)
        swap_index = zero_index - 1
        new_state[zero_index], new_state[swap_index] = new_state[swap_index], new_state[zero_index]
        neighbors.append(tuple(new_state))

    # Kanan
    if col < 2:
        new_state = list(state)
        swap_index = zero_index + 1
        new_state[zero_index], new_state[swap_index] = new_state[swap_index], new_state[zero_index]
        neighbors.append(tuple(new_state))

    return neighbors

### Greedy Best First Search

Greedy Best First Search memilih node berdasarkan nilai heuristik terkecil.

Fungsi evaluasi:

`f(n) = h(n)`

GBFS hanya mempertimbangkan jarak node menuju goal dan tidak menggunakan actual cost g(n) sebagai dasar prioritas.

In [21]:
def greedy_best_first_search(start, goal):
    visited = set()
    priority_queue = []
    h_start = manhattan_distance(start, goal)

    heapq.heappush(
        priority_queue,
        (h_start, start, [start])
    )

    while priority_queue:
        h, current, path = heapq.heappop(priority_queue)
        
        if current == goal:
            return path
        
        if current in visited:
            continue

        visited.add(current)

        for neighbor in get_neighbors(current):
            if neighbor not in visited:
                h_neighbor = manhattan_distance(
                    neighbor,
                    goal
                )

                heapq.heappush(
                    priority_queue,
                    (
                        h_neighbor,
                        neighbor,
                        path + [neighbor]
                    )
                )

    return None

### A*

Algoritma A* menggunakan kombinasi actual cost dan heuristic.

Fungsi evaluasi:

`f(n) = g(n) + h(n)`

dengan:

- g(n) = cost dari initial state menuju state saat ini.
- h(n) = Manhattan Distance menuju goal.
- f(n) = total nilai evaluasi.

Karena setiap perpindahan memiliki cost 1, maka nilai g(n) sama dengan jumlah langkah yang telah dilakukan.

In [22]:
def a_star_search(start, goal):
    priority_queue = []
    h_start = manhattan_distance(start, goal)

    heapq.heappush(
        priority_queue,
        (
            h_start,
            0,
            start,
            [start]
        )
    )

    best_cost = {
        start: 0
    }

    while priority_queue:
        f, g, current, path = heapq.heappop(priority_queue)

        if current == goal:
            return path

        if g > best_cost.get(current, float("inf")):
            continue

        for neighbor in get_neighbors(current):
            new_g = g + 1

            if new_g < best_cost.get(
                neighbor,
                float("inf")
            ):

                best_cost[neighbor] = new_g
                h = manhattan_distance(
                    neighbor,
                    goal
                )

                new_f = new_g + h
                heapq.heappush(
                    priority_queue,
                    (
                        new_f,
                        new_g,
                        neighbor,
                        path + [neighbor]
                    )
                )

    return None

### Cek Solvability

In [23]:
def count_inversions(state):
    numbers = [x for x in state if x != 0]
    inversions = 0

    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):

            if numbers[i] > numbers[j]:
                inversions += 1

    return inversions

def is_solvable(start, goal):
    start_inversions = count_inversions(start)
    goal_inversions = count_inversions(goal)

    return start_inversions % 2 == goal_inversions % 2

### Antarmuka Tkinter

Program menyediakan antarmuka grafis menggunakan **Tkinter**. User dapat:
1. Mengisi konfigurasi **Initial State** dan **Goal State** dengan angka 0–8.
2. Memilih algoritma **Greedy Best First Search** atau **A\***.
3. Menjalankan pencarian melalui tombol **Solve**.
4. Melihat setiap state pada rute solusi, nilai heuristik/evaluasi, dan total cost.
5. Menggunakan tombol **Reset** untuk mengembalikan contoh konfigurasi awal.

Setiap perpindahan memiliki **actual cost = 1** dan heuristik yang digunakan adalah **Manhattan Distance**.


In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox


class EightPuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Solver - GBFS & A*")
        self.root.geometry("1050x760")
        self.root.minsize(900, 650)

        self.default_start = (0, 1, 3, 4, 2, 5, 7, 8, 6)
        self.default_goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)

        self.start_entries = []
        self.goal_entries = []

        self.build_interface()
        self.reset_board()

    def build_interface(self):
        title = ttk.Label(
            self.root,
            text="8-PUZZLE SOLVER",
            font=("Poppins", 20, "bold")
        )

        title.pack(pady=(15, 2))
        subtitle = ttk.Label(
            self.root,
            text="Greedy Best First Search / A*",
            font=("Poppins", 10)
        )

        subtitle.pack(pady=(0, 12))
        input_frame = ttk.Frame(self.root)
        input_frame.pack(pady=5)

        self.create_board(
            input_frame,
            "Initial State",
            self.start_entries,
            0
        )

        self.create_board(
            input_frame,
            "Goal State",
            self.goal_entries,
            1
        )

        control_frame = ttk.Frame(self.root)
        control_frame.pack(pady=15)

        ttk.Label(
            control_frame,
            text="Algoritma:"
        ).grid(row=0, column=0, padx=5)

        self.algorithm_var = tk.StringVar(
            value="Greedy Best First Search"
        )

        algorithm_box = ttk.Combobox(
            control_frame,
            textvariable=self.algorithm_var,
            state="readonly",
            width=30,
            values=[
                "Greedy Best First Search",
                "A*"
            ]
        )

        algorithm_box.grid(row=0, column=1, padx=5)

        ttk.Button(
            control_frame,
            text="Solve",
            command=self.solve
        ).grid(row=0, column=2, padx=5)

        ttk.Button(
            control_frame,
            text="Reset",
            command=self.reset_board
        ).grid(row=0, column=3, padx=5)

        result_frame = ttk.LabelFrame(
            self.root,
            text="Hasil Pencarian"
        )

        result_frame.pack(
            fill="both",
            expand=True,
            padx=20,
            pady=(5, 20)
        )

        self.result_text = tk.Text(
            result_frame,
            wrap="none",
            font=("Consolas", 10)
        )

        vertical_scroll = ttk.Scrollbar(
            result_frame,
            orient="vertical",
            command=self.result_text.yview
        )
        
        horizontal_scroll = ttk.Scrollbar(
            result_frame,
            orient="horizontal",
            command=self.result_text.xview
        )

        self.result_text.configure(
            yscrollcommand=vertical_scroll.set,
            xscrollcommand=horizontal_scroll.set
        )

        self.result_text.grid(
            row=0,
            column=0,
            sticky="nsew"
        )

        vertical_scroll.grid(
            row=0,
            column=1,
            sticky="ns"
        )

        horizontal_scroll.grid(
            row=1,
            column=0,
            sticky="ew"
        )

        result_frame.rowconfigure(0, weight=1)
        result_frame.columnconfigure(0, weight=1)

    def create_board(self, parent, title, entries, column):
        frame = ttk.LabelFrame(
            parent,
            text=title,
            padding=12
        )

        frame.grid(
            row=0,
            column=column,
            padx=25
        )

        for i in range(9):
            entry = ttk.Entry(
                frame,
                width=4,
                justify="center",
                font=("Poppins", 14)
            )

            entry.grid(
                row=i // 3,
                column=i % 3,
                padx=4,
                pady=4,
                ipady=5
            )

            entries.append(entry)

    def set_board(self, entries, state):
        for entry, value in zip(entries, state):
            entry.delete(0, tk.END)
            entry.insert(0, str(value))

    def reset_board(self):
        self.set_board(
            self.start_entries,
            self.default_start
        )

        self.set_board(
            self.goal_entries,
            self.default_goal
        )

        self.algorithm_var.set(
            "Greedy Best First Search"
        )

        self.result_text.delete("1.0", tk.END)

    def read_board(self, entries, board_name):
        try:
            values = [int(entry.get().strip()) for entry in entries]

        except ValueError:
            raise ValueError(
                f"{board_name} HARUS BERISI ANGKA 0 SAMPAI 8."
            )

        if sorted(values) != list(range(9)):
            raise ValueError(
                f"{board_name} HARUS BERISI ANGKA 0 SAMPAI 8 DAN SETIAP ANGKA HANYA BOLEH MUNCUL 1 KALI."
            )

        return tuple(values)

    def solve(self):
        try:
            start = self.read_board(
                self.start_entries,
                "INITIAL STATE"
            )

            goal = self.read_board(
                self.goal_entries,
                "GOAL STATE"
            )

        except ValueError as error:
            messagebox.showerror(
                "INPUT TIDAK VALID",
                str(error)
            )

            return

        if not is_solvable(start, goal):
            self.result_text.delete("1.0", tk.END)
            self.result_text.insert(
                tk.END,
                "PUZZLE TIDAK DAPAT DISELESAIKAN\n"
                "INITIAL STATE DAN GOAL STATE MEMILIKI PARITAS INVERSI YANG BERBEDA"
            )

            return

        algorithm = self.algorithm_var.get()

        if algorithm == "Greedy Best First Search":
            path = greedy_best_first_search(start, goal)

        else:
            path = a_star_search(start, goal)

        self.show_result(
            path,
            goal,
            algorithm
        )

    def show_result(self, path, goal, algorithm):
        self.result_text.delete("1.0", tk.END)

        self.result_text.insert(
            tk.END,
            "=" * 70 + "\n"
        )

        self.result_text.insert(
            tk.END,
            "                          HASIL PENCARIAN\n"
        )

        self.result_text.insert(
            tk.END,
            "=" * 70 + "\n"
        )

        self.result_text.insert(
            tk.END,
            f"Algoritma    : {algorithm}\n"
        )

        total_cost = len(path) - 1

        self.result_text.insert(
            tk.END,
            f"Jumlah state : {len(path)}\n"
            f"Total langkah: {total_cost}\n"
            f"Total cost   : {total_cost}\n\n"
            f"RUTE SOLUSI\n"
            f"{'-' * 70}\n"
        )

        for step, state in enumerate(path):
            g = step
            h = manhattan_distance(
                state,
                goal
            )
            f = g + h

            self.result_text.insert(
                tk.END,
                f"Step {step}\n"
            )

            self.result_text.insert(
                tk.END,
                f"{state[0]} {state[1]} {state[2]}\n"
                f"{state[3]} {state[4]} {state[5]}\n"
                f"{state[6]} {state[7]} {state[8]}\n"
            )

            if algorithm == "Greedy Best First Search":
                self.result_text.insert(
                    tk.END,
                    f"h(n) = {h}\n"
                )
            else:
                self.result_text.insert(
                    tk.END,
                    f"g(n) = {g}, h(n) = {h}, f(n) = {f}\n"
                )

            if step < len(path) - 1:
                self.result_text.insert(
                    tk.END,
                    "Cost langkah berikutnya = 1\n"
                )

            self.result_text.insert(
                tk.END,
                f"{'-' * 70}\n"
            )

        self.result_text.insert(
            tk.END,
            f"\nTotal Cost = {total_cost}\n"
        )

root = tk.Tk()
app = EightPuzzleGUI(root)
root.mainloop()
